In [2]:
# ========= DATA PROFILE + PLOTS =========
import os, re, math, json, warnings
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


warnings.filterwarnings("ignore")

DATA_DIR = r"C:\Users\GSI\Pictures\DATASETS"
TRAIN_INTERACTIONS = os.path.join(DATA_DIR, "train_interactions.csv")
GAMES_FILE = os.path.join(DATA_DIR, "games.csv")
PLOT_DIR = DATA_DIR
os.makedirs(PLOT_DIR, exist_ok=True)

def _detect_cols(df):
    cols = {c.lower(): c for c in df.columns}
    u = cols.get("user_id") or cols.get("user") or cols.get("userid") or cols.get("steamid")
    i = cols.get("item_id") or cols.get("item") or cols.get("game_id") or cols.get("appid") or cols.get("app_id")
    p = cols.get("playtime") or cols.get("play_time") or cols.get("playtime_forever") or cols.get("rating") or cols.get("label")
    return u, i, p

def _parse_tokens(series, label):
    # phrase-preserving split on [ , | / ; ] only
    DELIMS_RE = re.compile(r"[\,\|/;]+")
    def norm(s):
        s = str(s).strip().lower()
        s = re.sub(r"\s+", " ", s).replace("-", "_").replace(" ", "_")
        s = re.sub(r"[^a-z0-9_]+", "", s)
        return s
    all_toks = []
    missing = 0
    for raw in series.fillna(""):
        if not raw:
            missing += 1
            continue
        parts = [p for p in DELIMS_RE.split(str(raw)) if p and p.strip()]
        toks = [norm(p) for p in parts if norm(p)]
        if toks:
            all_toks.extend(toks)
        else:
            missing += 1
    return Counter(all_toks), missing

print("=== DATA PROFILE ===")
inter = pd.read_csv(TRAIN_INTERACTIONS, low_memory=False)
u_col, i_col, p_col = _detect_cols(inter)
inter = inter.rename(columns={u_col:"user", i_col:"item"})
if p_col:
    inter["playtime"] = inter[p_col].astype(float)
else:
    inter["playtime"] = 1.0
inter = inter.dropna(subset=["user","item"])
inter["user"] = inter["user"].astype(int)
inter["item"] = inter["item"].astype(int)
inter = inter.drop_duplicates(["user","item"], keep="last")

n_int = len(inter)
n_users = inter["user"].nunique()
n_items = inter["item"].nunique()
density = 100.0 * n_int / (n_users * n_items)
avg_hist = inter.groupby("user")["item"].nunique().mean()
avg_pop = inter.groupby("item")["user"].nunique().mean()

print(f"Interactions: {n_int:,} | Users: {n_users:,} | Items: {n_items:,} | Density: {density:.4f}%")
print(f"Avg. items per user: {avg_hist:.2f} | Avg. users per item: {avg_pop:.2f}")

# Cold-start stats (consistent with later random_split convention: 80/10/10)
from libreco.data import random_split
tmp_train, tmp_eval, tmp_test = random_split(inter[["user","item"]], multi_ratios=[0.8, 0.1, 0.1], seed=42)
u_train, i_train = set(tmp_train["user"]), set(tmp_train["item"])
u_test, i_test = set(tmp_test["user"]), set(tmp_test["item"])
cold_users = len([u for u in u_test if u not in u_train])
cold_items = len([it for it in i_test if it not in i_train])
print(f"Cold users in test: {cold_users:,} / {len(u_test):,} | Cold items in test: {cold_items:,} / {len(i_test):,}")

# Token coverage from games.csv
games = pd.read_csv(GAMES_FILE, low_memory=False)
cg = {c.lower(): c for c in games.columns}
g_item = cg.get('item_id') or cg.get('appid') or cg.get('app_id') or cg.get('id')
g_tags = cg.get('tags') or cg.get('tag')
g_genres = cg.get('genres') or cg.get('genre')
games = games.rename(columns={g_item:"item", g_tags:"tags", g_genres:"genres"})
games["item"] = games["item"].astype(int)

in_catalog = set(inter["item"])
games = games[games["item"].isin(in_catalog)]
tag_counts, miss_tags = _parse_tokens(games["tags"] if "tags" in games else pd.Series([]), "tags")
genre_counts, miss_gen = _parse_tokens(games["genres"] if "genres" in games else pd.Series([]), "genres")
unique_tokens = len(set(tag_counts.keys()) | set(genre_counts.keys()))
print(f"Items with missing tags: {miss_tags:,} | missing genres: {miss_gen:,} | unique tokens (tags+genres): {unique_tokens:,}")

# ========== Plots ==========
# 1) User activity (items per user): long-tail
hist_sizes = inter.groupby("user")["item"].nunique()
plt.figure(figsize=(7,4))
plt.hist(hist_sizes.values, bins=100)
plt.yscale("log")
plt.xlabel("Items per user")
plt.ylabel("Count (log scale)")
plt.title("User activity distribution")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "profile_user_activity.png")); plt.close()

# 2) Item popularity (users per item): long-tail
item_pop = inter.groupby("item")["user"].nunique()
plt.figure(figsize=(7,4))
plt.hist(item_pop.values, bins=100)
plt.yscale("log")
plt.xlabel("Users per item")
plt.ylabel("Count (log scale)")
plt.title("Item popularity distribution")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "profile_item_popularity.png")); plt.close()

# 3) Playtime distribution (per interaction)
plt.figure(figsize=(7,4))
plt.hist(np.clip(inter["playtime"].values, 0, np.percentile(inter["playtime"].values, 99)), bins=100)
plt.xlabel("Playtime (clipped at 99th pct)")
plt.ylabel("Count")
plt.title("Playtime distribution")
plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "profile_playtime.png")); plt.close()

# 4) Top tokens bar
top_tokens = (tag_counts + genre_counts).most_common(20)
if top_tokens:
    labels, counts = zip(*top_tokens)
    plt.figure(figsize=(8,5))
    plt.bar(range(len(labels)), counts)
    plt.xticks(range(len(labels)), labels, rotation=45, ha="right")
    plt.ylabel("Doc frequency (#items having token)")
    plt.title("Top 20 tokens (tags+genres)")
    plt.tight_layout(); plt.savefig(os.path.join(PLOT_DIR, "profile_top_tokens.png")); plt.close()

print("Saved: profile_user_activity.png, profile_item_popularity.png, profile_playtime.png, profile_top_tokens.png")

=== DATA PROFILE ===
Interactions: 2,267,389 | Users: 54,315 | Items: 8,368 | Density: 0.4989%
Avg. items per user: 41.75 | Avg. users per item: 270.96
Cold users in test: 0 / 42,071 | Cold items in test: 0 / 5,760
Items with missing tags: 4 | missing genres: 404 | unique tokens (tags+genres): 341
Saved: profile_user_activity.png, profile_item_popularity.png, profile_playtime.png, profile_top_tokens.png
